#### Set environment variables in [.env](.env) for LLM API calling

### Import Dependencies

In [ ]:
import sys
import os
sys.path.insert(0, "../")
import promptwizard
import random
from promptwizard.glue.promptopt.instantiate import GluePromptOpt
from typing import Any
from tqdm import tqdm
import json

from dotenv import load_dotenv
load_dotenv(override=True)


### Prepare the BBH task-specific dataset processor

In [ ]:
from bbh_processor import BBH

bbh_processor = BBH()


### Load and save the dataset\nSet the `dataset_to_run` variable to choose 1 among the BBH tasks to run the optimization on

In [ ]:
dataset_to_run = 'hyperbaton'
seed = 42

# Shared, idempotent three-way split (data_prep.py) -- every optimizer
# notebook for this (task, seed) gets the identical train/mpir-validation/test
# partition, which is required for the example-level paired analysis in
# REBUILD.md §3.2 to compare like with like.
from data_prep import prepare_bbh_task_split

split_paths = prepare_bbh_task_split(dataset_to_run, bbh_processor, seed=seed)


### Set paths

In [ ]:
train_file_name = split_paths.train_file_name
test_file_name = split_paths.test_file_name
path_to_config = "configs"
promptopt_config_path = os.path.join(path_to_config, "protegi/promptopt_config.yaml")
setup_config_path = os.path.join(path_to_config, "protegi/setup_config.yaml")


### Create an object for calling prompt optimization and inference functionalities

In [ ]:
gp = GluePromptOpt(promptopt_config_path,
                   setup_config_path,
                   train_file_name,
                   bbh_processor,
                   seed=seed)


### Call prompt optimization function\nProTeGi runs beam search over textual-gradient edits of `base_instruction` (set in configs/protegi/promptopt_config.yaml), so `use_examples`/`run_without_train_examples`/`generate_synthetic_examples` are not meaningful here and are left at their defaults.

In [ ]:
best_prompt = gp.get_best_prompt()


### Save the optimized prompt and expert profile

In [ ]:
print(best_prompt[0])


In [ ]:
import pickle
if not os.path.exists("results"):
    os.mkdir("results")

# Task+seed-scoped so a downstream MPIR pass over this output can never
# silently load the wrong task/seed's prompt (see MPIR.ipynb / promptwizard.ipynb).
pkl_path = f"results/protegi_{dataset_to_run}_seed{seed}.pkl"
with open(pkl_path, 'wb') as f:
    pickle.dump(best_prompt[0], f)

print(f"Best prompt (saved to {pkl_path}): {best_prompt[0]}")


### Evaluate the optimized prompt

In [ ]:
gp.BEST_PROMPT = best_prompt[0]

# Function call to evaluate the prompt. task_name/condition_name/seed feed the
# tracked results/predictions/<task>_<condition>_<seed>.jsonl file (REBUILD.md §5).
accuracy = gp.evaluate(test_file_name, task_name=dataset_to_run,
                       condition_name="protegi", seed=seed)

print(f"Final Accuracy: {accuracy}")
